In [40]:
import pandas as pd
import numpy as np

clients_df = pd.read_csv('../data/Clients.csv')
property_df = pd.read_csv('../data/Properties.csv')

In [41]:
print("Client IDs:")
print(clients_df['client_id'].head(10))

print("\nProperty Client References:")
print(property_df['client_ref'].dropna().head(10))

Client IDs:
0    C0001
1    C0002
2    C0003
3    C0004
4    C0005
5    C0006
6    C0007
7    C0008
8    C0009
9    C0010
Name: client_id, dtype: str

Property Client References:
0    C0027
1    C0097
2    C0113
3    C0141
4    C0146
5    C0023
6    C0040
7    C0007
8    C0082
9    C0038
Name: client_ref, dtype: str


In [42]:
print("Unique clients:", clients_df['client_id'].nunique())
print("Unique property client references:", property_df['client_ref'].nunique())

Unique clients: 2000
Unique property client references: 2000


In [43]:
property_client_ids = set(property_df['client_ref'].dropna())
client_ids = set(clients_df['client_id'])

unmatched_ids = property_client_ids - client_ids

print("Unmatched property client references:", len(unmatched_ids))

Unmatched property client references: 0


In [44]:
property_df['client_ref'].value_counts().describe()

count    2000.0000
mean        3.6525
std         0.8397
min         3.0000
25%         3.0000
50%         4.0000
75%         4.0000
max        13.0000
Name: count, dtype: float64

integration + feature engineering

In [45]:
print("Property shape:", property_df.shape)
print("Client shape:", clients_df.shape)

Property shape: (10000, 9)
Client shape: (2000, 12)


In [46]:
merged_df = clients_df.merge(
    property_df,
    how = 'left',
    right_on = 'client_ref',
    left_on = 'client_id',
    
)

In [47]:
merged_df.shape

(7305, 21)

In [48]:
merged_df.head(10)

,client_id,client_type,first_name,last_name,date_of_birth,gender,country,region,acquisition_purpose,satisfaction_score,...,referral_channel,listing_id,tower_number,transaction_date,unit_category,unit_number,floor_area_sqft,sale_price,listing_status,client_ref
0,C0001,Individual,Kareem,Liu,05-11-1968,F,USA,California,Home,4,...,Website,90343,9,10-01-2024,Apartment,40,1090.32,"$351,419.29",Sold,C0001
1,C0001,Individual,Kareem,Liu,05-11-1968,F,USA,California,Home,4,...,Website,4051,4,12-01-2024,Apartment,51,1608.84,"$496,266.41",Sold,C0001
2,C0001,Individual,Kareem,Liu,05-11-1968,F,USA,California,Home,4,...,Website,150099,15,05-01-2025,Apartment,15,522.71,"$175,599.90",Sold,C0001
3,C0001,Individual,Kareem,Liu,05-11-1968,F,USA,California,Home,4,...,Website,30432,3,12-01-2025,Apartment,50,713.67,"$223,479.12",Sold,C0001
4,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,...,Website,150044,15,01-01-2024,Apartment,6,938.57,"$299,245.20",Sold,C0002
5,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,...,Website,1045,1,02-01-2024,Apartment,45,756.21,"$248,525.12",Sold,C0002
6,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,...,Website,90285,9,12-01-2024,Apartment,36,1582.79,"$505,127.63",Sold,C0002
7,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,...,Website,200450,20,05-01-2025,Apartment,51,1062.33,"$336,892.39",Sold,C0002
8,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,...,Website,30377,3,12-01-2025,Apartment,46,1599.81,"$451,305.59",Sold,C0002
9,C0003,Individual,Kale,Gay,04-07-1959,M,USA,California,Home,4,...,Agency,10104,1,07-01-2024,Apartment,13,719.47,"$216,874.97",Sold,C0003


In [49]:
merged_df[['floor_area_sqft', 'sale_price', 'transaction_date']].info()

<class 'pandas.DataFrame'>
RangeIndex: 7305 entries, 0 to 7304
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   floor_area_sqft   7305 non-null   float64
 1   sale_price        7305 non-null   str    
 2   transaction_date  7305 non-null   str    
dtypes: float64(1), str(2)
memory usage: 322.9 KB


In [50]:
merged_df[['floor_area_sqft', 'sale_price', 'transaction_date']].head()

,floor_area_sqft,sale_price,transaction_date
0,1090.32,"$351,419.29",10-01-2024
1,1608.84,"$496,266.41",12-01-2024
2,522.71,"$175,599.90",05-01-2025
3,713.67,"$223,479.12",12-01-2025
4,938.57,"$299,245.20",01-01-2024


In [52]:
merged_df['sale_price'] = (
    merged_df['sale_price']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

In [56]:
merged_df['transaction_date'] = pd.to_datetime(
    merged_df['transaction_date'],
    format='mixed',
    dayfirst=False,
    errors='coerce'
)

In [57]:
merged_df[['floor_area_sqft', 'sale_price', 'transaction_date']].info()

<class 'pandas.DataFrame'>
RangeIndex: 7305 entries, 0 to 7304
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   floor_area_sqft   7305 non-null   float64       
 1   sale_price        7305 non-null   float64       
 2   transaction_date  7305 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(2)
memory usage: 171.3 KB


In [55]:
merged_df[['floor_area_sqft', 'sale_price', 'transaction_date']].head()

,floor_area_sqft,sale_price,transaction_date
0,1090.32,351419.29,2024-10-01
1,1608.84,496266.41,2024-12-01
2,522.71,175599.90,2025-05-01
3,713.67,223479.12,2025-12-01
4,938.57,299245.20,2024-01-01
